In [ ]:
import os
import json
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import chromadb

# Embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Local Chroma client & collection
client = chromadb.Client(Settings(
    persist_directory="./chroma_db",  # folder to save DB locally
    anonymized_telemetry=False
))
collection_name = "my_study_docs"

if collection_name in [c.name for c in client.list_collections()]:
    collection = client.get_collection(collection_name)
else:
    collection = client.create_collection(name=collection_name)

# 3. Folder containing JSON chunk files
PREPROCESSED_FOLDER = "preprocessed"

def load_chunks_from_json(file_path):
    """Load chunks and metadata from one JSON file."""
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # Expecting data to be like: {"file_path": "...", "chunks": ["text1", "text2", ...]}
    # Adjust keys if your JSON structure differs
    return data.get("file_path", os.path.basename(file_path)), data.get("chunks", [])

def embed_and_add_to_chroma(file_path, chunks):
    embeddings = model.encode(chunks, show_progress_bar=True)
    ids = [f"{file_path}_{i}" for i in range(len(chunks))]  # unique IDs per chunk
    metadatas = [{"source_file": file_path, "chunk_index": i} for i in range(len(chunks))]
    documents = chunks  # raw text chunks

    collection.add(
        ids=ids,
        embeddings=embeddings.tolist() if hasattr(embeddings, "tolist") else embeddings,
        metadatas=metadatas,
        documents=documents,
    )

def main():
    # Loop over all JSON files in preprocessed folder
    for filename in os.listdir(PREPROCESSED_FOLDER):
        if not filename.endswith(".json"):
            continue
        full_path = os.path.join(PREPROCESSED_FOLDER, filename)
        file_path, chunks = load_chunks_from_json(full_path)
        if not chunks:
            print(f"No chunks found in {filename}, skipping.")
            continue
        print(f"Embedding and adding {len(chunks)} chunks from {file_path}...")
        embed_and_add_to_chroma(file_path, chunks)

    # Persist DB to disk
    client.persist()
    print("Chroma DB persisted to disk.")

ModuleNotFoundError: No module named 'sentence_transformers'